# Distributed Joins and Complex Aggregations

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 05_joins_and_aggregations.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete reference on distributed join types (inner, left, right, full, cross, semi, anti) and advanced aggregations (cube, rollup, pivot).
- **Why does this topic exist?**: Joins and aggregations are the most compute- and network-intensive operations in distributed data engineering.
- **What problem does it solve?**: Incorrect join types lead to duplicate rows, dropped data, or massive Cartesian products.
- **Where is it used in real-world Data Engineering?**: Denormalizing star-schema fact and dimension tables in data warehousing.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | DataFrames, Spark SQL, partitions |
| **Required depth** | Deep working knowledge: can choose join strategies and diagnose shuffle/aggregation issues |
| **Data Engineer** | Write correct joins/aggregations and inspect the chosen strategy. |
| **Senior Data Engineer** | Tune broadcast thresholds, join order, and skew handling. |
| **Spark specialist** | Explain join selection, exchange, hash build, sort-merge, and skew mitigation. |
| **Next topic** | `03_dataframe_and_spark_sql/06_window_functions.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Joins and aggregations are where distributed systems earn their keep — and where most expensive shuffles
happen. A join that is correct on 1,000 rows can explode on 100 million rows because of duplicate keys, nulls, or
a bad strategy. The optimizer chooses a strategy from statistics and hints; your job is to verify the choice.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Join type** | Which rows survive | Inner, left, right, full, semi, anti, cross |
| **Broadcast join** | Send small side to every executor | No shuffle of the large side |
| **Sort-merge join** | Shuffle and sort both sides | Default for large-large equi-joins |
| **Shuffle hash join** | Shuffle and build a hash map | Can be memory-intensive |
| **Aggregation** | Group and combine | Partial + final aggregation |
| **Skew** | Uneven key distribution | Stragglers and OOM |

### How it works, step by step

1. The optimizer estimates sizes from statistics/catalog.
2. It chooses broadcast, sort-merge, shuffle hash, or nested-loop join.
3. A shuffle exchange partitions both sides by join key.
4. The reduce side builds a hash table or merges sorted streams.
5. Aggregations combine partially on the map side when possible.
6. AQE can coalesce partitions and split skewed join partitions at runtime.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Distributed Join Types

```
+-----------------------------------------------------------------------+
| JOIN TYPES IN SPARK                                                   |
+-------------------+---------------------------------------------------+
| Inner Join        | Keeps matching rows in both tables.               |
| Left / Right Join | Keeps all rows from left/right + matching right/left|
| Full Outer Join   | Keeps all rows from both tables, filling nulls.   |
| Left Semi Join    | Filters left rows that have a match in right.     |
|                   | (Columns from right are NOT added).               |
| Left Anti Join    | Filters left rows that have NO match in right.    |
| Cross Join        | Cartesian product (M x N rows) - USE CAUTION!     |
+-------------------+---------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Joins_Aggregations_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Hands-On: Three-Table Join (Orders + Customers + Products)

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
customers = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/customers.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/products.csv")

# Join orders with customers (left join) and products (left join)
enriched_orders = (
    orders
    .join(customers, on="customer_id", how="left")
    .join(products, on="product_id", how="left")
    .select(
        "order_id", "order_ts", "customer_id", "first_name", "country", "region",
        "product_id", "product_name", "category", "quantity", "unit_price", "status"
    )
)

enriched_orders.show(3)

## 4. Left Anti Join: Finding Orphan Customers

Find customers who have never placed an order using Left Anti Join.

In [ ]:
# Customers with ZERO orders
dormant_customers = (
    customers
    .join(orders, on="customer_id", how="left_anti")
    .select("customer_id", "first_name", "email", "signup_date")
)

print(f"Total customers: {customers.count()}")
print(f"Dormant customers with 0 orders: {dormant_customers.count()}")
dormant_customers.show(3)

## 5. Advanced Aggregations: `rollup`, `cube`, and `pivot`

In [ ]:
# Pivot: Total sales by Region across Product Categories
pivot_df = (
    enriched_orders
    .filter("status = 'Completed'")
    .groupBy("region")
    .pivot("category")
    .agg(F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2))
    .na.fill(0.0)
)

print("Sales Pivot Table (Region vs Category):")
pivot_df.show()

---

## Experiment: Left Semi Join vs Inner Join Row Count and Performance

- **1. What are we testing?**: Prove that Left Semi Join does not duplicate left table rows even if the right table has 1-to-many matches.
- **2. What do we expect?**: Left Semi Join returns at most 1 copy of left rows and does not project right table columns.

In [ ]:
# Left semi join: find customers who placed at least 1 order
semi_df = customers.join(orders, on="customer_id", how="left_semi")

print(f"Total customers:                 {customers.count()}")
print(f"Customers with >= 1 order (semi): {semi_df.count()}")
print("Columns in semi join result:", semi_df.columns)

**3. What happened?**: semi_df contains only customers columns; no duplicates occurred despite customers having multiple orders.  
**4. Why did it happen?**: Left semi join tests existence of a match; it stops after finding the first match and outputs the left row.  
**5. Practical Engineering Lesson**: Use left_semi instead of inner join + distinct when filtering a table based on existence in another table.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Order enrichment with a product dimension

- **Industry**: E-commerce
- **Scenario**: A retailer joins a large orders fact table to a small product catalog.
- **Problem**: A sort-merge join shuffles the large fact table even though the dimension is tiny.
- **Why this concept is relevant**: A broadcast hash join sends the catalog to every executor and avoids shuffling orders.
- **Implementation idea**: Use `broadcast(products)` or a hint, verify the plan, and monitor executor memory.
- **Expected result**: The same enriched result with less network traffic.
- **Alternative**: A warehouse star-schema join with a broadcast optimization.
- **Trade-offs**: Spark broadcast is simple; a warehouse may manage distribution automatically.

### Use Case 2: Fraud aggregation by account and day

- **Industry**: Banking
- **Scenario**: A risk team aggregates transactions by account and day to compute velocity features.
- **Problem**: A few accounts have millions of transactions, creating skew in the reduce stage.
- **Why this concept is relevant**: Partial aggregation and AQE skew handling can reduce data before the final shuffle.
- **Implementation idea**: Pre-aggregate per partition, use `spark.sql.adaptive.skewJoin.enabled`, and inspect task duration percentiles.
- **Expected result**: Balanced aggregation with bounded memory.
- **Alternative**: A streaming feature store for real-time velocity.
- **Trade-offs**: Batch Spark is simple for historical features; streaming is needed for live decisions.

---

## Business Value

- Correct join strategy can cut runtime and network cost.
- Better skew handling improves SLA reliability.
- Fewer shuffle bytes can reduce infrastructure spend.
- Validated join cardinality protects revenue and risk reporting.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Check join cardinality before and after; duplicates can multiply rows.
- Broadcast only genuinely small tables; monitor executor memory.
- Filter both sides before the join.
- Handle null keys explicitly; nulls do not match in equi-joins.
- Use AQE and inspect the final plan; statistics can be stale.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Broadcast vs sort-merge plan

**Steps**

1. Join orders to products with a broadcast hint and inspect `explain()`.
2. Join again with a merge hint and inspect the exchanges.
3. Compare shuffle write in the Stages tab.

**What to observe**: Broadcast avoids shuffling the large side; sort-merge shuffles both sides.

**Healthy result**: The plan matches the intended strategy and memory is safe.

**Problematic result**: Broadcasting a table that is too large causes executor OOM.

**Engineering takeaway**: Verify the strategy and the size assumption.

---

## Trade-offs and Alternatives

### Advantages

- Rich join/aggregation API
- Optimizer chooses strategies
- AQE handles runtime skew

### Limitations

- Shuffles are expensive
- Statistics can be missing/stale
- Duplicate keys can explode results

### When to use it

- Star-schema enrichment
- Customer 360
- Fraud features
- Reporting

### When not to use it

- Cross joins on large data
- Broadcasting large tables
- Joining on skewed null keys

### Alternatives inside Spark

- Broadcast hints
- AQE skew join
- Bucketing

### Alternatives outside Spark

- Warehouse joins
- Streaming joins in Flink/Kafka Streams

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Multi-Column Aggregations with agg()
**Problem Statement**: Compute total revenue, avg unit price, and unique order count per product category.  
**Input / Context**: enriched_orders

In [ ]:
# Exercise 1: Category metrics

### Exercise 2 (Intermediate): Rollup for Hierarchical Subtotals
**Problem Statement**: Use rollup('region','channel') to compute subtotals at the region level and overall grand total.  
**Input / Context**: enriched_orders

In [ ]:
# Exercise 2: Rollup subtotals

### Exercise 3 (Advanced): Detect Products with Zero Sales
**Problem Statement**: Use a Left Anti Join between products and orders to identify all products that have never been purchased.  
**Input / Context**: products and orders

In [ ]:
# Exercise 3: Products with zero orders

---

## Production Debugging Exercise

### Join explosion

**Scenario**: A join produces 10x more rows than expected.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check duplicate keys on both sides.
2. Compare counts before and after the join.
3. Use `dropDuplicates` or a pre-aggregation if duplicates are unintended.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Multi-Column Aggregations with agg()):
```python
enriched_orders.groupBy("category").agg(
    F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_rev"),
    F.round(F.avg("unit_price"), 2).alias("avg_price"),
    F.countDistinct("order_id").alias("unique_orders")
).show()
```

#### Solution 2 (Rollup for Hierarchical Subtotals):
```python
enriched_orders.rollup("region","channel").agg(
    F.count("order_id").alias("orders_count"),
    F.round(F.sum(F.col("quantity")*F.col("unit_price")), 2).alias("revenue")
).orderBy(F.col("region").asc_nulls_last(), F.col("channel").asc_nulls_last()).show(10)
```

#### Solution 3 (Detect Products with Zero Sales):
```python
unsold_products = products.join(orders, on="product_id", how="left_anti")
print("Unsold products count:", unsold_products.count())
unsold_products.select("product_id","product_name","category").show(5)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Join Types**: Know when to use inner, left, left_semi, and left_anti.
- **Left Semi Efficiency**: Filters for existence without duplicating rows or pulling right columns.
- **Pivot & Rollup**: Powerful multidimensional aggregation primitives built into Spark SQL.

### Essential Technical Terms
- `Left Semi Join`: Returns left rows matching right table without right columns.
- `Left Anti Join`: Returns left rows with NO match in right table.
- `Pivot`: Rotates distinct column values into horizontal column headers.

### Common Engineering Mistakes
- **Unintentional Cross Joins**: Missing join conditions generate massive Cartesian products.
- **Duplicate Column Names**: Joining two tables with matching column names without disambiguation causes ambiguous reference errors.

### Interview Questions on This Topic
- **Q: What is the difference between Left Semi and Inner Join?**
  - *A*: Left semi only checks for existence, does not duplicate left rows on 1:N matches, and does not include right table columns.
- **Q: How do you resolve ambiguous column names after a join?**
  - *A*: Drop the column prior to join, or join on a string name ('id') rather than expression (df1.id == df2.id).


### More Interview Questions

- **Q: What are the main join strategies in Spark?**
  - *A*: Broadcast hash join, sort-merge join, shuffle hash join, and broadcast nested-loop join for non-equi/cross joins.
- **Q: Why can a join produce more rows than either input?**
  - *A*: Duplicate keys on both sides create a Cartesian product per key.

### What to Study Next
Open 03_dataframe_and_spark_sql/06_window_functions.ipynb to master analytical window functions.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')